In [1]:
import numpy as np
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import Select

In [12]:
class StocksScraper:
    def __init__(self, driver, timeout = 10):
        self.driver = driver
        self.wait = WebDriverWait(self.driver, timeout = timeout)
        self.data = []

    # function to check if webpage is fully loaded
    def wait_for_page_to_load(self, old_url=None):
        """
        Waits for the URL to change (if old_url is provided) and then 
        waits for the document.readyState to be complete.
        """
        # 1. If we are coming from another page, wait for the URL to actually change first
        if old_url:
            try:
                self.wait.until(EC.url_changes(old_url))
            except:
                print("The URL did not change within the timeout period.")
    
        # 2. Now wait for the new page to report 'complete' status
        try:
            self.wait.until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
        except:
            print(f"The page \"{self.driver.title}\" didn't fully load within the duration\n")
        else:
            # After the wait, driver.title and driver.current_url will be updated
            print(f"Successfully moved to: \"{self.driver.title}\"")
            print(f"Current URL: {self.driver.current_url}\n")
            
    def access_url(self, url):
        self.driver.get(url)
        self.wait_for_page_to_load(old_url = self.driver.current_url)

    def access_most_active_stocks(self):
        actions = ActionChains(self.driver)
        market_menu = self.wait.until(
            EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/a[1]/div[1]"))
        )
        actions.move_to_element(market_menu).perform()
        
        # Navigate and locate stocks
        stocks_element = self.wait.until(
            EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]/span[1]"))
        )
        actions.move_to_element(stocks_element).perform()
        
        most_active = self.wait.until(
            EC.element_to_be_clickable((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[1]/a[1]/span[1]"))
        )
        actions.move_to_element(most_active).perform()
        most_active.click()
        
        self.wait_for_page_to_load(old_url = self.driver.current_url)

    def extract_stocks_data(self):
        # extract data from webpage
        while True:
            # scraping
            self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = self.driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            print(len(rows))
        
            for row in rows:
                values = row.find_elements(By.TAG_NAME, "td")
                # print([val.text for val in values])
        
                stock = {
                    "name": values[1].text,
                    "symbol": values[0].text,
                    "price": values[3].text,
                    "change": values[4].text,
                    "volume": values[6].text,
                    "market_cap": values[8].text,
                    "pe_ratio": values[9].text
                }
                self.data.append(stock)
            # break
            # click next
            try:
                next_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]'))
                )
            except:
                print("The \"next\" button is not clickable. We have navigated through all the pages")
                break
            else:
                next_button.click()
                time.sleep(1)
    def clean_and_save_data(self, filename = "temp"):
        stocks_df = (
                        pd
                        .DataFrame(self.data)
                        .apply(lambda col: col.str.strip() if (col.dtype == "str") else col)
                        .assign(
                            price = lambda df_: pd.to_numeric(df_.price),
                            change = lambda df_: pd.to_numeric(df_.change.str.replace("+", "")),
                            volume = lambda df_: pd.to_numeric(df_.volume.str.replace("M", "")),
                            market_cap = lambda df_: df_.market_cap.apply(lambda val: float(val.replace("B", "")) if ("B" in val) else float(val.replace("T", "")) * 1000),
                            pe_ratio = lambda df_: (
                                df_
                                .pe_ratio
                                .replace("--", np.nan)
                                .str.replace(",", "")
                                .pipe(lambda col: pd.to_numeric(col))
                            )
                        )
                        .rename(columns = {
                                "price": "price_usd",
                                "volume": "volume_M",
                                "market_cap": "market_cap_B"
                            }
                        )
                        
                    )
        stocks_df.to_csv(f"{filename}.csv", index = False)

In [13]:
if __name__ == "__main__":
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    url = "https://finance.yahoo.com/"
    scraper = StocksScraper(driver, 5)
    
    scraper.access_url(url)
    scraper.access_most_active_stocks()
    scraper.extract_stocks_data()
    scraper.clean_and_save_data("yahoo_finance-stocks")

    driver.quit()

The URL did not change within the timeout period.
Successfully moved to: "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News"
Current URL: https://finance.yahoo.com/

Successfully moved to: "Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance"
Current URL: https://finance.yahoo.com/markets/stocks/most-active/

25
25
25
25
25
25
25
25
25
4
The "next" button is not clickable. We have navigated through all the pages
